# ML-11 — FlyRank Capstone Experiment

**Research question:** On a client-held-out evaluation split, does a simple Logistic Regression ranker prioritize pages with an observed decline label more effectively than the Week-4 rule?

This notebook is a reproducible capstone scaffold. Run cells from top to bottom in Google Colab. It does not claim results until you run it on the approved starter CSV and inspect the generated metrics.

**Evidence boundary:** the target is an observed decline proxy from the dataset snapshot. Scores support human review; they do not predict future decline or prove that refreshing a page will improve performance.

Repository: https://github.com/Sai875/flyrank-ml-internship

## 1. Setup and data access

Use the same starter dataset as Weeks 4–5: `data/raw/content_refresh_anonymized.csv`. The dataset must not be committed to Git. If it is not available in the runtime, upload the approved CSV from your local machine when prompted.

In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

# Try the repository checkout first. The dataset may be excluded from Git by design.
if not DATA_PATH.exists() and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload the approved content_refresh_anonymized.csv file. Do not upload secrets.")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_names:
        raise FileNotFoundError("No CSV was uploaded. Upload the approved starter CSV and rerun.")
    DATA_PATH = Path(csv_names[0])

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Starter CSV not found. In Colab, upload content_refresh_anonymized.csv when prompted, "
        "or place it at data/raw/content_refresh_anonymized.csv."
    )

df_raw = pd.read_csv(DATA_PATH)
required = {"client_id", "trend_direction", "days_since_last_update",
            "impressions_90d", "ctr", "avg_position", "content_type"}
missing = sorted(required - set(df_raw.columns))
if missing:
    raise ValueError(f"Starter CSV is missing required columns: {missing}")

print(f"Loaded {len(df_raw):,} rows from {DATA_PATH}")
print(f"Distinct clients: {df_raw['client_id'].nunique():,}")

## 2. Data contract and leakage controls

The evaluation label is derived from `trend_direction` for measurement only. Do not use `trend_direction`, `trend_pct`, identifiers, or recommendation-derived fields as model features. A zero `avg_position` means no position data, not rank zero.

In [ ]:
df = df_raw.copy()
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(np.int8)
)

df["avg_position_no_data"] = (pd.to_numeric(df["avg_position"], errors="coerce").fillna(0) == 0).astype(np.int8)
df["avg_position_model"] = pd.to_numeric(df["avg_position"], errors="coerce").replace(0, np.nan)
df["days_since_last_update"] = pd.to_numeric(df["days_since_last_update"], errors="coerce")
df["impressions_90d"] = pd.to_numeric(df["impressions_90d"], errors="coerce")
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")

if "position_tier" not in df.columns:
    def get_tier(pos):
        if pd.isna(pos) or pos == 0:
            return "no_data"
        if pos <= 3:
            return "top_3"
        if pos <= 10:
            return "top_10"
        return "other"
    df["position_tier"] = pd.to_numeric(df["avg_position"], errors="coerce").map(get_tier)

FEATURES_NUMERIC = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position_model",
    "avg_position_no_data",
]
FEATURES_CATEGORICAL = ["content_type"]
FORBIDDEN = {
    "is_declining_label", "trend_direction", "trend_pct",
    "content_id", "client_id", "action_score", "action_label",
}
assert not (set(FEATURES_NUMERIC + FEATURES_CATEGORICAL) & FORBIDDEN)

print(f"Observed decline-label rate: {df['is_declining_label'].mean():.3f}")
print("Forbidden feature overlap:", set(FEATURES_NUMERIC + FEATURES_CATEGORICAL) & FORBIDDEN)
display(df[["is_declining_label", *FEATURES_NUMERIC, *FEATURES_CATEGORICAL]].describe(include="all").T)

## 3. Evaluation design

Use a fixed 80/20 client-grouped split (seed 42). A client must not appear in both train and test. The Week-4 rule's position-tier CTR median is estimated from the training clients only, then applied to the held-out clients. Both methods are evaluated on the exact same held-out rows with Precision@K.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(
    splitter.split(df, y=df["is_declining_label"], groups=df["client_id"])
)
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

train_clients = set(train["client_id"])
test_clients = set(test["client_id"])
assert train_clients.isdisjoint(test_clients), "Client leakage detected!"

print(f"Train rows: {len(train):,} | test rows: {len(test):,}")
print(f"Train clients: {len(train_clients)} | test clients: {len(test_clients)}")
print(f"Overlapping clients: {len(train_clients & test_clients)}")
print(f"Train label rate: {train['is_declining_label'].mean():.3f}")
print(f"Test label rate: {test['is_declining_label'].mean():.3f}")

## 4. Train the model and reproduce the baseline

Baseline rule: rank pages only when they are stale (90+ days), visible (500+ impressions in 90 days), and below the training-set median CTR for their position tier. Among flagged pages, rank by impressions. The learned model is Logistic Regression with median imputation/scaling for numeric features and one-hot encoding for content type.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Week-4 rule, calibrated only on the training split.
pos_median_ctr = (
    train.loc[train["position_tier"] != "no_data"]
    .groupby("position_tier")["ctr"]
    .median()
)
expected_ctr = test["position_tier"].map(pos_median_ctr)
stale = (test["days_since_last_update"] >= 90).astype(np.int8)
visible = (test["impressions_90d"] >= 500).astype(np.int8)
ctr_underperform = (
    (test["position_tier"] != "no_data") & (test["ctr"] < expected_ctr)
).astype(np.int8)
baseline_score = stale * visible * ctr_underperform * test["impressions_90d"].fillna(0)

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, FEATURES_NUMERIC),
    ("cat", categorical_pipe, FEATURES_CATEGORICAL),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(max_iter=1000, random_state=SEED)),
])

X_train = train[FEATURES_NUMERIC + FEATURES_CATEGORICAL]
X_test = test[FEATURES_NUMERIC + FEATURES_CATEGORICAL]
y_train = train["is_declining_label"]
y_test = test["is_declining_label"].to_numpy()

model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    k = min(int(k), len(labels))
    order = np.argsort(-np.asarray(scores), kind="mergesort")[:k]
    return float(np.asarray(labels)[order].mean())

ks = [10, 20, 50, 100]
results = []
for k in ks:
    results.append({
        "k": k,
        "metric": f"Precision@{k}",
        "Week-4 baseline": precision_at_k(baseline_score, y_test, k),
        "Logistic Regression": precision_at_k(model_score, y_test, k),
        "test label rate": float(np.mean(y_test)),
    })

comparison = pd.DataFrame(results)
display(comparison.round(3))

## 5. Results and chart

Interpret the comparison using the same held-out rows and metric. Do not write that the model is better unless the observed metrics support that conclusion. Precision@K is the fraction of the top K ranked pages carrying the observed decline label; it is not a causal or future-performance measure.

In [ ]:
ax = comparison.set_index("metric")[["Week-4 baseline", "Logistic Regression"]].plot(
    kind="bar", figsize=(9, 5), ylim=(0, 1), rot=0
)
ax.set_ylabel("Precision@K")
ax.set_xlabel("")
ax.set_title("Held-out clients: baseline vs Logistic Regression")
ax.legend(loc="best")
plt.tight_layout()
figure_path = Path("work/figures/capstone_precision_at_k.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved chart to {figure_path}")

## 6. Limitations and error review

- The label is an observed snapshot proxy, not a verified future decline outcome.
- The dataset may contain client-specific measurement differences and missingness.
- A grouped split tests generalization to held-out clients within this dataset, not all future clients or time periods.
- Precision@K is sensitive to the label base rate and the chosen K.
- Scores are decision support for human review; they do not establish that content changes cause improved traffic.
- Inspect false positives and missed declines before recommending deployment.

In [ ]:
# Aggregate-only error review. Never print or publish client/content identifiers or row-level examples.
review_results = pd.DataFrame({
    "is_declining_label": np.asarray(y_test, dtype=int),
    "model_score": np.asarray(model_score, dtype=float),
})
k = min(100, len(review_results))
top_k = review_results.nlargest(k, "model_score")
true_positives = int(top_k["is_declining_label"].sum())
false_positives = int(k - true_positives)
print(f"Aggregate error review for top {k} model-ranked rows:")
print(f"Observed declines in top {k}: {true_positives}")
print(f"Non-declines in top {k}: {false_positives}")
print("Row-level examples and identifiers are intentionally not displayed or saved.")

## 7. Reproducibility receipts

Save compact aggregate metrics as JSON. Do not save or commit the raw dataset, client identifiers, content identifiers, or a row-level review queue. Commit only the aggregate metrics and report-safe figures.

In [ ]:
from datetime import datetime, timezone

metrics = {
    "experiment": "capstone_client_grouped_holdout",
    "seed": SEED,
    "split": "GroupShuffleSplit test_size=0.20 grouped by client_id",
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "overlapping_clients": int(len(train_clients & test_clients)),
    "test_label_rate": float(np.mean(y_test)),
    "features_numeric": FEATURES_NUMERIC,
    "features_categorical": FEATURES_CATEGORICAL,
    "metrics": comparison.to_dict(orient="records"),
    "caveat": "Observed decline proxy; decision support only; no causal claims.",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
metrics_path = out_dir / "capstone_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved aggregate metrics to {metrics_path}")
print("Before committing, inspect the JSON and ensure it contains no identifiers or row-level data.")

### Automatic GitHub upload (one-time secret setup)

Preferred setup: add a Colab Secret named `GITHUB_TOKEN` and enable notebook access. Use a fine-grained GitHub token limited to this repository with **Contents: Read and write**. If the secret is missing, the upload cell now asks for the token using a hidden input instead of silently skipping. Never paste a token into notebook code or chat.


In [ ]:
# Automatically upload only the aggregate metrics JSON and report-safe chart.
# Preferred: configure Colab Secret GITHUB_TOKEN with notebook access enabled.
# Fallback: if the secret is missing, enter a GitHub token in the hidden prompt.
import base64
import getpass
import requests

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

if not github_token:
    print("GITHUB_TOKEN was not found in Colab Secrets.")
    print("Enter a fine-grained GitHub token with Contents: Read and write access to Sai875/flyrank-ml-internship.")
    print("The token input is hidden and will not be printed or saved in the notebook.")
    github_token = getpass.getpass("GitHub token (input hidden): ").strip()

if not github_token:
    raise RuntimeError("No GitHub token provided. Add GITHUB_TOKEN in Colab Secrets or rerun this cell and enter a token.")

repo = "Sai875/flyrank-ml-internship"
branch = "main"
files_to_upload = {
    "work/outputs/capstone_metrics.json": Path("work/outputs/capstone_metrics.json"),
    "work/figures/capstone_precision_at_k.png": Path("work/figures/capstone_precision_at_k.png"),
}
headers = {
    "Authorization": f"Bearer {github_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
session = requests.Session()
session.headers.update(headers)
try:
    for repo_path, local_path in files_to_upload.items():
        if not local_path.is_file():
            raise FileNotFoundError(f"Required output is missing: {local_path}. Run the metrics and chart cells first.")
        with local_path.open("rb") as f:
            encoded_content = base64.b64encode(f.read()).decode("ascii")
        api_url = f"https://api.github.com/repos/{repo}/contents/{repo_path}"
        check = session.get(api_url, params={"ref": branch}, timeout=30)
        if check.status_code == 200:
            existing_sha = check.json()["sha"]
        elif check.status_code == 404:
            existing_sha = None
        else:
            raise RuntimeError(f"GitHub check failed for {repo_path}: HTTP {check.status_code} — {check.text[:300]}")
        payload = {"message": f"Update {repo_path}", "content": encoded_content, "branch": branch}
        if existing_sha:
            payload["sha"] = existing_sha
        response = session.put(api_url, json=payload, timeout=60)
        if response.status_code not in (200, 201):
            raise RuntimeError(f"GitHub upload failed for {repo_path}: HTTP {response.status_code} — {response.text[:300]}")
        print(f"Uploaded successfully: {repo_path}")
    print("All report-safe capstone outputs uploaded to GitHub.")
finally:
    session.close()
    del github_token, headers, session


GITHUB_TOKEN was not found in Colab Secrets.
Enter a fine-grained GitHub token with Contents: Read and write access to Sai875/flyrank-ml-internship.
The token input is hidden and will not be printed or saved in the notebook.


## 8. Ranked recommendations

Use the results above to rank recommendations, and fill in the evidence rather than inventing claims:

1. **Human review queue:** use the higher-performing method at the K that matches the team's actual review capacity, if the improvement is consistent in the measured results.
2. **Audit data availability:** treat missing or zero-coded analytics carefully; distinguish unavailable data from true zero performance.
3. **Reduce concentration risk:** inspect whether a small number of clients dominate the top-ranked pages; consider per-client review caps only as a separately evaluated follow-up.
4. **Validate over time:** evaluate on a later sealed period before using the score operationally.
5. **Keep a human in the loop:** a score prioritizes investigation; it is not an automatic content-refresh decision.

## 9. Acknowledgments and data credit

This educational project uses FlyRank internship materials. Credit and project context: [FlyRank](https://flyrank.ai). Follow the repository's `DATA_USE.md` and public-safety guidance. Do not publish client identifiers, content identifiers, raw data, tokens, or row-level examples.

## ML-12 closing deliverables

### Five-minute demo outline
1. State the research question and the observed-label limitation.
2. Show the client-grouped split and leakage checks.
3. Compare baseline and Logistic Regression Precision@K on the same held-out rows.
4. Explain one limitation and one human-review recommendation.
5. Show the committed aggregate metrics JSON and research paper.

### Social-post cut
“I evaluated a simple content-review ranking approach against a rule-based baseline using a client-held-out split. The results are directional decision-support evidence, not causal proof or a guarantee of future traffic outcomes.”

### Employer-facing summary
Built a reproducible ML evaluation workflow with leakage controls, group-based validation, baseline comparison, Precision@K reporting, error review, and public-safe aggregate receipts. Replace this summary with the measured results after running the notebook.